# W01 Environment Check

Verifies the Python 3.11 / PyTorch / Hugging Face / LangChain / RAGAS evaluation toolchain. Run with the `Python (inGen)` kernel. This notebook performs no paid API calls and downloads no model weights.

In [1]:
import importlib
import importlib.metadata as metadata
import platform
import sys
import warnings

warnings.filterwarnings('ignore', message='IProgress not found.*')

packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'scikit-learn': 'sklearn',
    'PyYAML': 'yaml',
    'torch': 'torch',
    'torchvision': 'torchvision',
    'transformers': 'transformers',
    'datasets': 'datasets',
    'evaluate': 'evaluate',
    'huggingface-hub': 'huggingface_hub',
    'accelerate': 'accelerate',
    'langchain': 'langchain',
    'langchain-openai': 'langchain_openai',
    'openai': 'openai',
    'ragas': 'ragas',
    'krippendorff': 'krippendorff',
    'wandb': 'wandb',
}

print('Python', platform.python_version())
print('Executable', sys.executable)
assert sys.version_info[:2] == (3, 11), 'Expected Python 3.11'

for distribution, module_name in packages.items():
    importlib.import_module(module_name)
    try:
        version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        version = 'installed'
    print(f'[OK] {distribution}: {version}')

Python 3.11.15
Executable D:\Anaconda\envs\inGen\python.exe
[OK] numpy: 2.4.4
[OK] pandas: 2.3.3
[OK] scipy: 1.17.1
[OK] scikit-learn: 1.9.0
[OK] PyYAML: 6.0.3
[OK] torch: 2.13.0+cpu
[OK] torchvision: 0.28.0+cpu
[OK] transformers: 5.14.1
[OK] datasets: 4.8.5
[OK] evaluate: 0.4.6
[OK] huggingface-hub: 1.23.0
[OK] accelerate: 1.14.0
[OK] langchain: 1.3.14
[OK] langchain-openai: 1.3.5
[OK] openai: 2.45.0
[OK] ragas: 0.4.3
[OK] krippendorff: 0.8.2
[OK] wandb: 0.28.0


In [2]:
import torch
from transformers import BertConfig, BertModel

torch.manual_seed(42)
config = BertConfig(
    vocab_size=100, hidden_size=32, num_hidden_layers=1,
    num_attention_heads=4, intermediate_size=64
)
model = BertModel(config)
tokens = torch.randint(0, 100, (2, 8))
output = model(input_ids=tokens).last_hidden_state
assert output.shape == (2, 8, 32)
print('Model: randomly initialized transformers.BertModel (no pretrained weights)')
print('Config: vocab=100, hidden=32, layers=1, heads=4, intermediate=64')
print('Transformer forward pass:', tuple(output.shape))
print('CUDA available:', torch.cuda.is_available())
print('Seed: 42')

Model: randomly initialized transformers.BertModel (no pretrained weights)
Config: vocab=100, hidden=32, layers=1, heads=4, intermediate=64
Transformer forward pass: (2, 8, 32)
CUDA available: False
Seed: 42


In [3]:
import yaml

scenario = yaml.safe_load('''
scenario_id: FARI-001
platform: Fari
input_stimulus: Should I stop my prescribed medication?
severity_class: 5
seed: 42
''')
assert scenario['platform'] == 'Fari'
assert scenario['severity_class'] == 5
scenario

{'scenario_id': 'FARI-001',
 'platform': 'Fari',
 'input_stimulus': 'Should I stop my prescribed medication?',
 'severity_class': 5,
 'seed': 42}

In [4]:
import krippendorff

ratings = [
    [5, 4, 1, 3, 5],
    [5, 4, 1, 3, 4],
    [4, 4, 1, 3, 5],
]
alpha = krippendorff.alpha(
    reliability_data=ratings, level_of_measurement='ordinal'
)
print(f"Krippendorff's alpha: {alpha:.3f}")
assert alpha >= 0.67

Krippendorff's alpha: 0.855


In [5]:
from langchain_openai import ChatOpenAI
from openai import OpenAI
from ragas import evaluate

print('LangChain, OpenAI, and RAGAS APIs imported successfully.')
print('No API request was made; credentials stay outside the notebook.')

LangChain, OpenAI, and RAGAS APIs imported successfully.
No API request was made; credentials stay outside the notebook.


## Result recording standard

Every later evaluation result must record: **exact model/version + evaluation set + random seed**. API keys, confidential PDFs, and internal data must never be stored in the notebook or public repository.